## RAG Pipelines - Data ingestion to Vector DB Pipeline

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from pathlib import Path

/tmp/ipykernel_23891/378936446.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader


In [2]:
def load_documents(source):
    dir_loader = DirectoryLoader(
        path=source,
        glob="*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )

    documents = dir_loader.load()
    print(f"{len(documents)} are loaded")
    return documents

In [3]:
documents = load_documents("/home/sonip/projects/agentic-ai/src/agentic_ai/updatedlangchain/rag/data")


100%|██████████| 2/2 [00:02<00:00,  1.00s/it]

506 are loaded


RecursiveCharacterTextSplitter splits long text into chunks of at most chunk_size characters, trying to keep semantically related text together.

It works by trying a list of separators in priority order, by default ["\n\n", "\n", " ", ""]:

1. Split on the first separator (paragraphs).
2. Merge the resulting pieces back together up to chunk_size.
3. If any piece is still too big, recurse on it with the next separator (lines, then words, then single characters).

So it prefers breaking at paragraph boundaries, then lines, then words, and only cuts mid-word as a last resort.

chunk_overlap repeats the tail of one chunk at the start of the next, so context isn't lost at the boundaries.

It's the usual default splitter for RAG pipelines because it keeps chunks close to the size limit without splitting sentences or paragraphs more than necessary.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def make_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunked_documents = text_splitter.split_documents(documents)
    print(f"{len(documents)} are chunked into {len(chunked_documents)} using chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
    return chunked_documents
    

In [5]:
chunked_documents = make_chunks(documents)

506 are chunked into 1632 using chunk_size=1000, chunk_overlap=200


In [6]:
chunked_documents[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '/home/sonip/projects/agentic-ai/src/agentic_ai/updatedlangchain/rag/data/attention-is-all-you-need.pdf', 'file_path': '/home/sonip/projects/agentic-ai/src/agentic_ai/updatedlangchain/rag/data/attention-is-all-you-need.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz

## Embedding and VectorDB

### `sentence_transformers`
sentence-transformers is a Python library that turns text (sentences, paragraphs, chunks) into dense numeric vectors called embeddings. Texts with similar meaning get vectors that are close together.

Uses:
- Semantic search / RAG: embed your document chunks and the user's query, then retrieve the closest chunks from a vector store (FAISS, Chroma, etc.). This is its role in your RAG pipeline.
- Similarity and deduplication: compare how close two texts are in meaning, not just in shared words.
- Clustering and classification: group or label texts using their embeddings.
- Reranking: cross-encoder models score query and document pairs for more accurate ordering.

It runs locally and for free. Models like `all-MiniLM-L6-v2` are small and fast, so you don't need an API such as OpenAI embeddings. In LangChain you usually reach it through HuggingFaceEmbeddings, which wraps it.

In [7]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

/home/sonip/projects/agentic-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Define embedding model
class EmbeddingManager:
    """Documents embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


In [9]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1600.39it/s]


Model loaded successfully. Embedding dimension: 384


In [10]:
import hashlib

class VectorStore:
    """Manages document embeddings in a ChromaDB vectore database"""
    def __init__(self, collection_name: str = "pdf_documents_cosine", persist_directory: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """initialize chromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG", "hnsw:space": "cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
        except Exception as e:
            print(f"Error initializing vector: {e}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and corresponding embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ...")
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            key = f"{doc.metadata.get('source', '')}|{doc.metadata.get('page', '')}|{i}|{doc.page_content}"
            doc_id = hashlib.md5(key.encode("utf-8")).hexdigest()
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # Upsert in a single call: re-running with the same chunks overwrites instead of duplicating
        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Added {len(ids)} documents. Total in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents_cosine


In [11]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunked_documents]

embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 1632 texts...


Batches: 100%|██████████| 51/51 [01:03<00:00,  1.24s/it]

Generated embeddings with shape: (1632, 384)


In [12]:
vector_store.add_documents(chunked_documents, embeddings)

Adding 1632 documents to vector store ...
Added 1632 documents. Total in collection: 1632


In [13]:
class RAGRetriever:
    """Handle query based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector score
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )
            retrieved_docs = list()
            # Process results

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, doc, meta, dist) in enumerate(zip(ids, documents, metadatas, distances)):
                    # ChromaDB uses cosine distance, so 1-dist is cosine similarity.
                    similarity_score = 1 - dist

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': doc,
                            'metadata': meta,
                            'similarity_score': similarity_score,
                            'distance': dist,
                            'rank': i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        


In [14]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [15]:
context = rag_retriever.retrieve("How to handle millions of active user in a application?")

Retrieving documents for query: 'How to handle millions of active user in a application?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.45it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


In [16]:
context

[{'id': 'e0168e9ecee0a080a4442fc8b28e99df',
  'content': 'and running a large number of queries in parallel could easily overwhelm the data‐\nbase [35].\nIn order to achieve good throughput in a batch process, the computation must be\nlocal to one machine as much as possible. Making random-access requests over the\nnetwork for every record you want to process is too slow. Moreover, querying a\nremote database would mean that the batch job becomes non-deterministic, because\nthe data in the remote database might change.\nThus, a better approach would be to take a copy of the users database (for example,\nextracted from a database backup using an ETL process, see “Data warehousing” on\npage 88) and to put it in the same distributed filesystem as the log of user activity\nevents. You then have the user database in one set of files in HDFS, and the user\nactivity records in another set of files. MapReduce can be used to bring together all of\nthe relevant records in the same place and proc

In [17]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
)


In [19]:
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question"

    prompt = f"""Use the following context to answer the question concisely.
        Context: 
        {context}

        Question:
        {query}
    """

    response = llm.invoke(prompt)
    return response.content

def rag_advanced(query, retriever, llm, top_k=3, min_score=0.2, return_context = False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])


    prompt = f"""Use the following context to answer the question concisely.
        Context: 
        {context}

        Question:
        {query}
    """

    response = llm.invoke(prompt)
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context

    return output
    

In [20]:
query = "How to handle millions of active user in a application?"
answer = rag_simple(query, rag_retriever, llm)

Retrieving documents for query: 'How to handle millions of active user in a application?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


In [21]:
print(answer)

**Handle millions of active users by making the data and the computation as distributed and local as possible:**

1. **Copy the user database to the same distributed storage as the activity logs** (e.g., HDFS).  
   * This eliminates per‑request remote DB look‑ups, removes network latency, and makes the job deterministic.

2. **Partition the data**:  
   * **Hash‑partition** users (or user‑id) across many nodes so reads/writes are spread evenly.  
   * Keep all events for a given user on a single partition, ordered by timestamp, to simplify joins and updates.

3. **Use batch‑oriented processing (MapReduce / Spark)** for heavy analytics:  
   * Map phase reads the activity logs and the local copy of the user profile files.  
   * Reduce (or a sort‑merge join) brings together each user’s profile with its events in‑memory on the same node, achieving high throughput.

4. **Mitigate hot‑spots and skew**:  
   * Detect skewed keys (e.g., very popular users) and route them to dedicated “hot‑s

In [23]:
query2 = "How to handle millions of active user in a application?"
answer2 = rag_advanced(query2, rag_retriever, llm)
print(answer2['answer'])

Retrieving documents for query: 'How to handle millions of active user in a application?'
Top K: 3, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Handle millions of active users by moving the workload off a single remote DB and into a distributed, data‑local processing pipeline:**

1. **Snapshot the user database** – periodically export the user table (e.g., from a backup) into flat files and store them in the same distributed file system (HDFS) that holds the activity logs.

2. **Co‑locate data** – keep the user‑profile files and the activity‑event files on HDFS so every compute node can read both locally; no per‑record network round‑trips to a remote DB.

3. **Use a batch‑oriented engine (MapReduce / Spark)** –  
   * **Map phase:** read activity records, emit `<user_id, activity>` pairs.  
   * **Shuffle/partition:** hash‑partition on `user_id` so all records for a given user end up on the same reducer, spreading load evenly.  
   * **Reduce phase:** perform a **sort‑merge join** between the partitioned activity stream and the corresponding partition of the user‑profile file (both are already sorted by `user_id`). This yiel